# 1. Find the broadcasts

Lists the interpreted news broadcasts of the three archives in `work/broadcasts.json`:

- **PrimaTV**: the FOCUS 22 and 23 videos (the evening news) of PrimaTV's YouTube channel. Whether
  a video's thumbnail shows the interpreter is checked by eye (`decisions.json`).
- **ProTV**: the morning news (Știrile PRO TV, 06:00), from protv.ro's archive and from the
  432p files of vid2.stirileprotv.ro, by date.
- **Digi24**: the 11:00 news journal (Jurnal ora 11.00) of digi24.ro's archive, by date, with
  the 720p video file of each page.

The release's Digi24 lists are in `frozen/broadcasts.json`. Its ProTV lists were not kept; each
ProTV broadcast's protv.ro episode is its `source` in `frozen/segments.json`. In 'update' mode
only what is needed for the days after the release's is fetched.

These are public servers: the requests are spaced (`RATE` per second).

In [ ]:
import re
import threading
import time
from datetime import date

import requests
from bs4 import BeautifulSoup

from common import CROP, DECISIONS, INTERPRETERS, MODE, WORK, Faces, last_day, read, write

assert MODE != 'replay', "1_find runs in 'live' or 'update' mode (common.MODE)"
RATE = 10                  # requests per second, at most
COOKIES = 'cookies.txt'    # Netscape cookies.txt of a logged-in YouTube session (PrimaTV)
OUT = WORK / 'broadcasts.json'
broadcasts = read(OUT) if OUT.exists() else {}
since = {c: last_day(c) if MODE == 'update' else '' for c in ('ProTV', 'Digi24')}   # the release's last day

_lock, _last = threading.Lock(), [0.0]


def get(session, url, method='get', **kw):
    """A request, at most RATE per second (the response, whatever its status)."""
    with _lock:
        wait = _last[0] + 1 / RATE - time.monotonic()
        if wait > 0:
            time.sleep(wait)
        _last[0] = time.monotonic()
    return session.request(method, url, timeout=30, headers={'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64; rv:140.0) Gecko/20100101 Firefox/140.0'}, **kw)


MONTHS = {m: i for i, m in enumerate('ianuarie februarie martie aprilie mai iunie iulie august septembrie '
                                     'octombrie noiembrie decembrie'.split(), 1)}


def romanian_date(text):
    """'5 martie 2024' (anywhere in text) -> '2024-03-05'."""
    m = re.search(r'(\d{1,2})\s+([a-zăâîșț]+)\s+(\d{4})', text.lower())
    return date(int(m[3]), MONTHS[m[2]], int(m[1])).isoformat() if m and m[2] in MONTHS else None

## PrimaTV

The videos titled FOCUS 22 or FOCUS 23 of the channel's FOCUS playlists and of its uploads (some
are in neither playlist), with titles in Romanian (YouTube translates some, and the translations
do not match). The first playlist reached YouTube's limit of 5,000 videos in January 2026; the
second goes on from there. For each video not checked yet, the interpreter box of its thumbnail and
of 6 frames sampled uniformly over the video (read from its 720p stream by seeking, so only small
parts are fetched; a thumbnail alone often misses the interpreter) go to face recognition of the
channel's interpreters. A video where none is recognized gets a review image for a human.

In [ ]:
import subprocess

import cv2
import numpy as np
from yt_dlp import YoutubeDL

SOURCES = ['https://www.youtube.com/playlist?list=PL4GqKMm44AA10WhcE1-ZFdZjhxXLXHHhU',    # FOCUS, up to January 2026
           'https://www.youtube.com/playlist?list=PL4GqKMm44AA29oD-amyJ5JIaq9c7mAme1',    # Focus Prima TV, from January 2026
           'https://www.youtube.com/channel/UC6Sn1XzRBCBl8UMyAb8_5PA/videos']            # the channel's uploads
THUMB_BOX = {720: (1020, 480, 221, 237), 360: (388, 225, 82, 88)}    # the interpreter box, by thumbnail height
FRAME_BOX = tuple(CROP['PrimaTV'][720].values())                     # and in a 720p frame: the released crop
REVIEW = WORK / 'review'
REVIEW.mkdir(parents=True, exist_ok=True)
decisions = read(DECISIONS)
primatv = decisions['PrimaTV']

listed = {}                                  # id -> title
with YoutubeDL({'quiet': True, 'skip_download': True, 'extract_flat': 'in_playlist', 'cookiefile': COOKIES,
                'extractor_args': {'youtube': {'lang': ['ro']}}}) as ydl:
    for source in SOURCES:
        for e in ydl.extract_info(source, download=False)['entries']:
            if e and re.match(r'^FOCUS\s*2[23]', e.get('title') or '', re.I):
                listed[e['id']] = e['title']
videos = sorted(listed)
recognized = broadcasts.get('PrimaTV', {}).get('recognized', {})    # per video checked: the interpreter seen, or None
broadcasts['PrimaTV'] = {'videos': videos, 'recognized': recognized}
write(OUT, broadcasts)
faces = Faces(INTERPRETERS['PrimaTV'])


def sampled_frames(vid, n=6):
    """n frames sampled uniformly over the video (None where one cannot be read)."""
    with YoutubeDL({'quiet': True, 'format': 'bv[height=720][vcodec^=avc1]/bv[height=720]', 'cookiefile': COOKIES,
                    'js_runtimes': {'node': {}}, 'remote_components': ['ejs:github']}) as ydl:
        info = ydl.extract_info(f'https://www.youtube.com/watch?v={vid}', download=False)
    frames = []
    for k in range(n):
        png = subprocess.run(['ffmpeg', '-v', 'error', '-ss', f"{info['duration'] * (k + 0.5) / n:.2f}", '-i', info['url'], '-frames:v', '1',
                              '-f', 'image2pipe', '-vcodec', 'png', '-'], capture_output=True).stdout
        frames.append(cv2.imdecode(np.frombuffer(png, np.uint8), cv2.IMREAD_COLOR) if png else None)
    return frames


session = requests.Session()
for vid in (v for v in videos if v not in primatv['thumbnails'] and v not in recognized):
    thumb = None
    for name in ('maxresdefault', 'hqdefault'):
        r = get(session, f'https://i.ytimg.com/vi/{vid}/{name}.jpg')
        if r.ok:
            thumb = cv2.imdecode(np.frombuffer(r.content, np.uint8), cv2.IMREAD_COLOR)
            break
    x, y, w, h = FRAME_BOX
    try:
        frames = sampled_frames(vid)
    except Exception as error:
        print(vid, 'not sampled, left for the next run:', str(error)[:200])
        continue
    tiles = [np.zeros((h, w, 3), np.uint8) if f is None or f.shape[0] != 720 else f[y:y + h, x:x + w] for f in frames]
    if thumb is not None:
        tx, ty, tw, th = THUMB_BOX[thumb.shape[0]]
        tiles.insert(0, cv2.resize(thumb[ty:ty + th, tx:tx + tw], (w, h)))
    seen = [n for n in (faces.who(np.ascontiguousarray(t[:, :, ::-1])) for t in tiles) if n]
    recognized[vid] = max(set(seen), key=seen.count) if seen else None
    if not seen:
        cv2.imwrite(str(REVIEW / f'{vid}.jpg'), np.hstack(tiles))
    write(OUT, broadcasts)
print(len(videos), 'videos,', sum(1 for v in recognized.values() if v), 'with a recognized interpreter,',
      len(list(REVIEW.glob('*.jpg'))), 'to check by eye')

The videos where no known interpreter was recognized: look at every review image (the thumbnail's
box first, then the 6 frames) and delete the ones that never show an interpreter: `d` next, `a`
previous, `x` delete, `q` done. The verdicts of the images shown (image left: an interpreter, known
or not; a new one needs a picture in `known_people/` and in `common.INTERPRETERS`) go to
`decisions.json` (PrimaTV, `thumbnails`), and override the recognition.

In [ ]:
files, i, seen = sorted(REVIEW.glob('*.jpg')), 0, set()
cv2.namedWindow('review', cv2.WINDOW_NORMAL)
while files:
    print(files[i].stem)
    seen.add(files[i].stem)
    cv2.imshow('review', cv2.imread(str(files[i])))
    key = cv2.waitKey(0)
    if key == ord('q'):
        break
    if key == ord('x'):
        files.pop(i).unlink()
        i %= max(len(files), 1)
    elif key in (ord('d'), ord('a')):
        i = (i + (1 if key == ord('d') else -1)) % len(files)
cv2.destroyAllWindows()

checked = {v: (REVIEW / f'{v}.jpg').exists() for v in seen}      # only the images shown
primatv['thumbnails'] = dict(sorted({**primatv['thumbnails'], **checked}.items()))
write(DECISIONS, decisions)

## ProTV

Two routes, as in the original run:

- **protv.ro** lists its episodes newest first, about a dozen per page; the morning news are
  titled "Stirile PRO TV # 06.00 – <date>". The original run took the broadcasts up to
  2024-02-29 from here (720p).
- **vid2.stirileprotv.ro** has the 432p files of the morning news listed on stirileprotv.ro,
  at `<date>/<id>-2.mp4`, where the date is the day the file was stored, which can differ from
  the day it aired: a missing file is looked for up to two years either side. The original run
  took the broadcasts from 2024-03-01 on from here. vid2 no longer serves its files (it answers
  HTTP 500), and stirileprotv.ro now lists only the last few months; then nothing is recorded
  for vid2.

In [ ]:
LISTING = 'https://www.protv.ro/emisiuni/stirile-pro-tv/videoclipuri/episoade/pagina-'
MORNING = re.compile(r'^\s*Stirile PRO TV\s*#\s*06[.:]00')
VID2_FROM = '2024-03-01'

session, protv, page = requests.Session(), {}, 1
while True:
    r = get(session, LISTING + str(page))
    if r.status_code != 200:
        break
    soup = BeautifulSoup(r.text, 'html.parser')
    links = [a for a in soup.find_all('a', href=True) if 'episodul' in a['href']]
    if not links:
        break
    days = []
    for a in links:
        title = a.get_text().strip()
        day = MORNING.match(title) and romanian_date(title)
        if day and day > since['ProTV'] and a['href'] not in protv.setdefault(day, []):
            protv[day].append(a['href'])
        days += [day] if day else []
    if days and max(days) <= since['ProTV']:     # newest first: the rest is done
        break
    page += 1
print(page - 1, 'protv.ro pages,', len(protv), 'days')

In [ ]:
from datetime import timedelta

vid2, page = {}, 1
listed = []
while True:
    r = get(session, f'https://stirileprotv.ro/video/stirile-diminetii/?page={page}')
    items = BeautifulSoup(r.text, 'html.parser').select('div.vid_item') if r.ok else []
    for item in items:
        a = item.find('a', href=True)
        m = re.search(r'(\d{2})\.(\d{2})\.(\d{4})', a.find(class_='title').get_text(strip=True))
        if m:
            listed.append((date(int(m[3]), int(m[2]), int(m[1])), re.search(r'\d+', a['href'])[0]))
    if not items or not listed or listed[-1][0].isoformat() < max(VID2_FROM, since['ProTV']):
        break
    page += 1

for aired, number in listed:
    if aired.isoformat() < VID2_FROM or aired.isoformat() <= since['ProTV']:
        continue
    for days in [0] + [d for i in range(1, 2 * 365 + 1) for d in (-i, i)]:
        stored = aired + timedelta(days)
        url = f"https://vid2.stirileprotv.ro/{stored.strftime('%Y/%m/%d')}/{number}-2.mp4"
        status = get(session, url, 'head').status_code
        if status == 200:
            vid2.setdefault(stored.isoformat(), []).append(url)
            break
        if status != 404:          # not "no such file": the server does not serve files
            break
    if status not in (200, 404):
        print('vid2 answers', status, '- not used')
        break
broadcasts['ProTV'] = {'protv.ro': dict(sorted(protv.items())), 'vid2': dict(sorted(vid2.items()))}
write(OUT, broadcasts)
print(len(listed), 'stirileprotv.ro broadcasts,', sum(map(len, vid2.values())), 'vid2 files')

## Digi24

digi24.ro lists its journals newest first. The 11:00 journal's articles are titled
"Jurnal ora 11.00 ..." and dated "Data publicării - <date>"; a date can have more than one.
Each article page carries its player's configuration, with the video file of every rendition.

In [ ]:
import json

DIGI = 'https://www.digi24.ro'
PLAYER_CFG = re.compile(r'<script[^>]+id="video-player-cfg"[^>]*>(.*?)</script>', re.S)

session, listing, page = requests.Session(), {}, 1
while True:
    r = get(session, f'{DIGI}/emisiuni/jurnale?p={page}', allow_redirects=False)
    if r.status_code != 200:          # past the last page the site redirects
        break
    for article in BeautifulSoup(r.text, 'html.parser').find_all('article'):
        a, em = article.select_one('h2 a'), article.find('em')
        if a and em and a.get_text(strip=True).startswith('Jurnal ora 11.00') \
                and em.get_text(strip=True).startswith('Data publicării - '):
            url = DIGI + a['href']
            day = romanian_date(em.get_text())
            if url not in listing.setdefault(day, []):
                listing[day].append(url)
    page += 1

video_src = {}
for day, urls in listing.items():
    if day <= since['Digi24']:
        continue
    for url in urls:
        cfg = PLAYER_CFG.search(get(session, url).text)
        versions = json.loads(cfg[1])['new-info']['meta'].get('versions') or {} if cfg else {}
        video_src[url] = versions.get('720p.mp4')
broadcasts['Digi24'] = {'listing': dict(sorted(listing.items())), 'video_src': video_src}
write(OUT, broadcasts)
print(len(listing), 'days,', sum(v is None for v in video_src.values()), 'pages without a 720p video')